# RFM Analysis & Customer Segmentation
### Papa John's — Upminster (Hubbun Ltd) | RQ2: Retention & Reactivation

This notebook continues from your data cleaning. It builds **Recency, Frequency and
Monetary** features, scores each customer 1–5, and assigns behavioural segments that map
directly onto your retention/reactivation recommendations.

> **If you already have your cleaned `df` in memory**, you can skip Cell 1 and start at
> "Step 1". Cell 1 just re-runs a condensed version of your cleaning so the notebook works
> stand-alone.


## Cell 1 — Load + (condensed) cleaning + capture the non-buyers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Same file as your cleaning notebook (first sheet = 'Customer Data')
df = pd.read_excel('modified data pj.xlsx')

# --- RQ1 evidence: record NON-BUYERS *before* you filter them out ---
never_ordered = int((df['Total Orders'] == 0).sum())
print(f"Registered customers who have NEVER ordered: {never_ordered} "
      f"({never_ordered/len(df)*100:.1f}% of the raw list)")
print("  -> keep this number: it is direct evidence for the acquisition problem (RQ1)\n")

# --- Your cleaning pipeline (condensed) ---
df = df.dropna(subset=['Phone'])
df['Source'] = df['Source'].fillna('Unknown')
df = df.drop(columns=['Email', 'Address'])
df['Phone'] = df['Phone'].astype(str).str.replace('.0', '', regex=False)
df.loc[df['Phone'].str.len() == 9, 'Phone'] = '0' + df.loc[df['Phone'].str.len() == 9, 'Phone']
df = df[df['Phone'].str.len() >= 10]
df['Last Order Date'] = pd.to_datetime(df['Last Order Date'])
df = df.sort_values('Last Order Date', ascending=False).drop_duplicates(subset='Phone', keep='first')
df = df[(df['Total Orders'] > 0) & (df['Avg Order Value (£)'] > 0)]
df['Customer_ID'] = ['CUST_' + str(i).zfill(4) for i in range(1, len(df) + 1)]
df = df.drop(columns=['Phone', 'ID'])

print(f"Clean buyers for RFM: {len(df)}")
df.head()


## Step 1 — Build the RFM features

| Feature | Definition | Note |
|---|---|---|
| **Recency** | days since last order | lower = more recent = better |
| **Frequency** | lifetime order count (`Total Orders`) | over the full ~13-month data window (state this as a limitation) |
| **Monetary** | estimated total spend = `Total Orders` × `Avg Order Value` | you only have AOV, not per-order spend, so this is an estimate |


In [ ]:
# Snapshot = day after the most recent order in the data
snapshot_date = df['Last Order Date'].max() + pd.Timedelta(days=1)
print("Snapshot (analysis) date:", snapshot_date.date())

rfm = df[['Customer_ID', 'Postcode', 'Source', 'Order Type']].copy()
rfm['Recency']   = (snapshot_date - df['Last Order Date']).dt.days
rfm['Frequency'] = df['Total Orders']
rfm['Monetary']  = (df['Total Orders'] * df['Avg Order Value (£)']).round(2)   # est. total spend
rfm['AOV']       = df['Avg Order Value (£)']                                    # keep for profiling

rfm[['Recency', 'Frequency', 'Monetary']].describe().round(1)


## Step 2 — Score each customer 1–5 (quintiles)

Recency is reversed (more recent → higher score). `rank(method='first')` breaks ties on
Frequency so the quintile bins don't collapse (integer order counts have lots of ties).


In [ ]:
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

rfm[['Recency', 'Frequency', 'Monetary', 'R', 'F', 'M', 'RFM_Score']].head()


## Step 3 — Assign behavioural segments

A standard Recency × Frequency segment map. The names translate straight into RQ2 actions:
**Champions/Loyal** = protect & reward, **At Risk / Can't Lose Them** = priority win-back,
**New / Promising** = nudge toward a 2nd/3rd order, **Lost/Hibernating** = low-cost reactivation.


In [ ]:
def assign_segment(r, f):
    if r >= 4 and f >= 4:  return 'Champions'
    if r >= 3 and f >= 3:  return 'Loyal'
    if r >= 4 and f <= 2:  return 'New / Promising'
    if r == 3 and f <= 2:  return 'Potential Loyalist'
    if r == 2 and f >= 3:  return 'At Risk'
    if r <= 2 and f >= 4:  return "Can't Lose Them"
    if r == 2 and f <= 2:  return 'Hibernating'
    if r <= 1:             return 'Lost'
    return 'Other'

rfm['Segment'] = [assign_segment(r, f) for r, f in zip(rfm['R'], rfm['F'])]

summary = (rfm.groupby('Segment')
              .agg(Customers=('Customer_ID', 'size'),
                   Avg_Recency=('Recency', 'mean'),
                   Avg_Frequency=('Frequency', 'mean'),
                   Avg_AOV=('AOV', 'mean'),
                   Total_Value=('Monetary', 'sum'))
              .sort_values('Total_Value', ascending=False))
summary['% of Customers'] = (summary['Customers'] / summary['Customers'].sum() * 100).round(1)
summary['% of Revenue']   = (summary['Total_Value'] / summary['Total_Value'].sum() * 100).round(1)
summary.round(1)


## Step 4 — Active vs lapsed (your core business question)

Define "active" as having ordered within the last N days. **Agree the exact threshold with
Hubbun** — 90 days is a sensible default for a takeaway.


In [ ]:
ACTIVE_DAYS = 90   # <-- agree this threshold with Hubbun

rfm['Status'] = np.where(rfm['Recency'] <= ACTIVE_DAYS, 'Active', 'Lapsed')
status = rfm['Status'].value_counts()
print(status.to_string())
print(f"\nActive customers: {status.get('Active', 0)} "
      f"({status.get('Active', 0) / len(rfm) * 100:.1f}% of buyers)")

# Revenue drifting away = the reactivation prize (RQ2)
at_risk = rfm[rfm['Segment'].isin(['At Risk', "Can't Lose Them"])]['Monetary'].sum()
print(f"Revenue tied to At-Risk + Can't-Lose-Them: "
      f"{at_risk / rfm['Monetary'].sum() * 100:.1f}% of total")


## Step 5 — Visualise the segments

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

summary['Customers'].plot(kind='barh', ax=ax[0], color='#C8102E')   # Papa John's red
ax[0].set_title('Customers per Segment'); ax[0].set_xlabel('Customers'); ax[0].invert_yaxis()

summary['% of Revenue'].plot(kind='barh', ax=ax[1], color='#00693E')  # Papa John's green
ax[1].set_title('Share of Estimated Revenue per Segment')
ax[1].set_xlabel('% of revenue'); ax[1].invert_yaxis()

plt.tight_layout(); plt.show()


## Save + what comes next

`rfm_segments.csv` becomes the input for the rest of your analysis:

1. **K-Means clustering** — scale `Recency, Frequency, Monetary`, use the elbow/silhouette
   method to pick *k*, then compare the data-driven clusters with these rule-based segments.
2. **Simple CLV** — `AOV × Frequency × est. margin` per segment, to prioritise spend.
3. **Pareto (80/20)** — cumulative revenue vs customers ranked by `Monetary`.
4. **Explanatory classifier** — logistic regression / decision tree predicting `Status`
   (Active vs Lapsed) from Frequency, AOV, Source, Order Type (**not** Recency — that defines
   the label). Frame it as *which customer traits are associated with lapsing*.


In [ ]:
rfm.to_csv('rfm_segments.csv', index=False)
print("Saved rfm_segments.csv —", rfm.shape[0], "customers,", rfm['Segment'].nunique(), "segments")
